In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [10]:
df = pd.read_csv("data/raw/diabetic_data.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [12]:
# 1. Inspect column names, data types, and non-null counts
df.info()

# 2. Check missing values, including '?' placeholders
df_check = df.replace("?", np.nan)

missing_summary = pd.DataFrame({
    "Missing Count": df_check.isna().sum(),
    "Missing Percentage": (
        df_check.isna().mean() * 100
    ).round(2)
})

display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ].sort_values("Missing Percentage", ascending=False)
)

# 3. Check duplicate encounter IDs
print("Duplicate encounters:",
      df.duplicated(subset=["encounter_id"]).sum())

# 4. Check unique patients
print("Unique patients:", df["patient_nbr"].nunique())

# 5. Examine readmission outcomes
print("\nReadmission categories:")
display(df["readmitted"].value_counts())

print("\nReadmission percentages:")
display(
    (df["readmitted"].value_counts(normalize=True) * 100)
    .round(2)
)

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

,Missing Count,Missing Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


Duplicate encounters: 0
Unique patients: 71518

Readmission categories:


readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


Readmission percentages:


readmitted
NO     53.91
>30    34.93
<30    11.16
Name: proportion, dtype: float64

In [17]:
# Identify missing-value placeholders
missing_summary = pd.DataFrame({
    "Missing Count": df_clean.replace("?", np.nan).isna().sum(),
    "Missing Percentage": (
        df_clean.replace("?", np.nan).isna().mean() * 100
    ).round(2)
})

display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ].sort_values("Missing Percentage", ascending=False)
)

,Missing Count,Missing Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [18]:
# Work on a copy of the original dataset
df_clean = df.copy()

# Replace '?' placeholders with missing values
df_clean = df_clean.replace("?", np.nan)

# Remove weight due to extremely high missingness
df_clean = df_clean.drop(columns=["weight"])

# Create indicators for recorded laboratory results
df_clean["a1c_result_recorded"] = (
    df_clean["A1Cresult"].notna().astype(int)
)

df_clean["glucose_result_recorded"] = (
    df_clean["max_glu_serum"].notna().astype(int)
)

# Preserve the distinction between missing and recorded results
df_clean["A1Cresult"] = df_clean["A1Cresult"].fillna("Not recorded")
df_clean["max_glu_serum"] = df_clean["max_glu_serum"].fillna("Not recorded")

# Fill missing categorical information
categorical_columns = [
    "medical_specialty",
    "payer_code",
    "race",
    "diag_1",
    "diag_2",
    "diag_3"
]

for col in categorical_columns:
    df_clean[col] = df_clean[col].fillna("Unknown")

# Create binary 30-day readmission outcome
df_clean["readmitted_30_days"] = (
    df_clean["readmitted"] == "<30"
).astype(int)

print("Cleaned dataset shape:", df_clean.shape)
print("Remaining missing values:", df_clean.isna().sum().sum())

Cleaned dataset shape: (101766, 52)
Remaining missing values: 0


In [19]:
from pathlib import Path

# Create a folder for processed data
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
df_clean.to_csv(
    "data/processed/diabetes_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [20]:
# Reload the cleaned dataset
df_saved = pd.read_csv(
    "data/processed/diabetes_cleaned.csv"
)

print("Saved dataset shape:", df_saved.shape)
print("Missing values:", df_saved.isna().sum().sum())

Saved dataset shape: (101766, 52)
Missing values: 0


In [28]:
# Display every column and its data type
pd.set_option("display.max_rows", 100)

column_check = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.values,
    "Unique Values": df.nunique().values,
    "Missing Values": df.isna().sum().values
})

display(column_check)

,Column,Data Type,Unique Values,Missing Values
0,encounter_id,int64,101766,0
1,patient_nbr,int64,71518,0
2,race,str,6,0
3,gender,str,3,0
4,age,str,10,0
5,admission_type_id,int64,8,0
6,discharge_disposition_id,int64,26,0
7,admission_source_id,int64,17,0
8,time_in_hospital,int64,14,0
9,payer_code,str,18,0
